# TWSE PPO + SMC Training Notebook

This notebook version mirrors `train_pipeline.py` and splits the training flow into editable steps.

## 1. Imports

In [ ]:
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

from twse_pipeline_common import (
    BASKET_CONFIG,
    PAIR_CONFIG,
    TWTradingEnv,
    build_feature_frame,
    download_ohlcv,
    ensure_model_dir,
)

## 2. Training Settings

Adjust these values before running the training cells. The original script uses `500_000` timesteps and `cuda:0`.

In [ ]:
TRAIN_START = "2018-01-01"
TRAIN_END = "2024-12-31"
TOTAL_TIMESTEPS = 500_000
INITIAL_BALANCE = 1_000_000.0
DEVICE = "cuda:0"

# Train both models by default, same as train_pipeline.py.
# You can change this to [PAIR_CONFIG] or [BASKET_CONFIG] for a shorter run.
CONFIGS_TO_TRAIN = [PAIR_CONFIG, BASKET_CONFIG]

## 3. CUDA Check

The original training script requires a CUDA GPU. Run this cell before preparing data or training.

In [ ]:
def require_cuda():
    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA GPU is required for training. "
            "Install a CUDA-enabled PyTorch build or run on a machine with an NVIDIA GPU."
        )
    torch.set_float32_matmul_precision("high")
    torch.backends.cudnn.benchmark = True


require_cuda()
print(f"Using fixed training device: {DEVICE}")
print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 4. Download Market Data

This cell downloads OHLCV data from yfinance for the training period. The raw data is kept in memory so later cells can build features without downloading again.

In [ ]:
raw_market_data = {}

for config in CONFIGS_TO_TRAIN:
    print(f"\n=== Downloading data for {config.name} ===")
    print(f"Tickers: {', '.join(config.tickers)}")
    raw_market_data[config.name] = download_ohlcv(config.tickers, TRAIN_START, TRAIN_END)
    for ticker, df in raw_market_data[config.name].items():
        print(f"{ticker}: {len(df)} rows, {df['date'].min().date()} to {df['date'].max().date()}")

## 5. Build Feature Frames

This converts the raw OHLCV data into the SMC and spread z-score features used by the PPO environment.

In [ ]:
feature_data = {}

for config in CONFIGS_TO_TRAIN:
    feature_df = build_feature_frame(raw_market_data[config.name], config)
    feature_data[config.name] = feature_df
    print(f"{config.name}: {len(feature_df)} feature rows, {len(feature_df.columns)} columns")

feature_data[CONFIGS_TO_TRAIN[0].name].head()

## 6. Training Function

In [ ]:
def train_one_model(config):
    print(f"\n=== Training {config.name} ===")
    print(f"Tickers: {', '.join(config.tickers)}")
    print(f"Period: {TRAIN_START} to {TRAIN_END}")

    if config.name in feature_data:
        feature_df = feature_data[config.name]
    else:
        raw_frames = download_ohlcv(config.tickers, TRAIN_START, TRAIN_END)
        feature_df = build_feature_frame(raw_frames, config)
    print(f"Feature rows: {len(feature_df)}")

    env = DummyVecEnv(
        [
            lambda: TWTradingEnv(
                feature_df=feature_df,
                config=config,
                initial_balance=INITIAL_BALANCE,
            )
        ]
    )

    policy_kwargs = dict(net_arch=dict(pi=[128, 128], vf=[128, 128]))
    model = PPO(
        "MlpPolicy",
        env,
        learning_rate=2e-4,
        n_steps=2048,
        batch_size=128,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.005,
        target_kl=0.03,
        policy_kwargs=policy_kwargs,
        verbose=1,
        device=DEVICE,
    )

    model.learn(total_timesteps=TOTAL_TIMESTEPS)
    ensure_model_dir(config.model_path)
    model.save(config.model_path)
    print(f"Saved model: {config.model_path}")
    return model

## 7. Start Training

This cell may take a long time. It saves models to the paths defined in `PAIR_CONFIG` and `BASKET_CONFIG`.

In [ ]:
trained_models = {}

for config in CONFIGS_TO_TRAIN:
    trained_models[config.name] = train_one_model(config)

trained_models.keys()